In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kiet222005/xes3g5m-externd/test.csv
/kaggle/input/datasets/kiet222005/xes3g5m-externd/test_quelevel.csv
/kaggle/input/datasets/kiet222005/xes3g5m-externd/test_question_sequences.csv
/kaggle/input/datasets/kiet222005/xes3g5m-externd/test_question_window_sequences.csv
/kaggle/input/datasets/kiet222005/xes3g5m-externd/test_sequences.csv
/kaggle/input/datasets/kiet222005/xes3g5m-externd/test_sequences_quelevel.csv
/kaggle/input/datasets/kiet222005/xes3g5m-externd/test_window_sequences.csv
/kaggle/input/datasets/kiet222005/xes3g5m-externd/test_window_sequences_quelevel.csv
/kaggle/input/datasets/kiet222005/xes3g5m-externd/train_valid_sequences.csv
/kaggle/input/datasets/kiet222005/xes3g5m-externd/train_valid_sequences_quelevel.csv


In [2]:
%%bash
set -e
cd /kaggle/working
[ -d pykt-toolkit ] || git clone https://github.com/pykt-team/pykt-toolkit.git
cd pykt-toolkit
pip install -q -e . && pip install -q einops matplotlib scipy tqdm

X=/kaggle/input/datasets/kiet222005/xes3g5m-externd
D=/kaggle/working/pykt-toolkit/data/peiyou
mkdir -p $D
for f in train_valid_sequences test_sequences test_window_sequences test_question_sequences test_question_window_sequences; do
  cp $X/$f.csv $D/
done
cp $X/*.json $D/ 2>/dev/null || true

cd examples && mkdir -p logs
date +"start train: %T"
python -u wandb_dkt_train.py --dataset_name=peiyou --fold=0 --seed=42 \
    --use_wandb=0 --add_uuid=0 --save_dir=sm_dkt > logs/dkt_f0_s42_train.log 2>&1
date +"end train: %T"
tail -5 logs/dkt_f0_s42_train.log
find sm_dkt -name "*.ckpt" -o -name "config.json"
df -h /kaggle/working | tail -1

start train: 09:23:05
end train: 09:28:30
Epoch: 18, validauc: 0.8347, validacc: 0.8415, best epoch: 8, best auc: 0.8347, train loss: 0.3439845944579058, emb_type: qid, model: dkt, save_dir: sm_dkt/peiyou_dkt_qid_sm_dkt_42_0_0.2_200_0.001_0_0
            testauc: -1, testacc: -1, window_testauc: -1, window_testacc: -1
fold	modelname	embtype	testauc	testacc	window_testauc	window_testacc	validauc	validacc	best_epoch
0	dkt	qid	-1	-1	-1	-1	0.834678886431592	0.8414548688653567	8
end:2026-10-22 09:28:26.974367
sm_dkt/peiyou_dkt_qid_sm_dkt_42_0_0.2_200_0.001_0_0/config.json
sm_dkt/peiyou_dkt_qid_sm_dkt_42_0_0.2_200_0.001_0_0/qid_model.ckpt
/dev/loop2       20G   12G  7.9G  60% /kaggle/working


Cloning into 'pykt-toolkit'...


In [3]:
%%bash
cd /kaggle/working/pykt-toolkit/examples
CKPT=$(dirname $(find sm_dkt -name "config.json" | head -n 1))
echo "CKPT=$CKPT"; free -h | head -2

python -u wandb_predict.py --save_dir=$CKPT --bz=64 --use_wandb=0 \
    --fusion_type=late_fusion > logs/dkt_f0_s42_latefusion.log 2>&1 &
PID=$!
while kill -0 $PID 2>/dev/null; do
  echo "$(date +%T) $(free -g | awk '/Mem/{print "RAM used: "$3"G / "$2"G"}')"; sleep 15
done
wait $PID; echo "exit code: $?"
tail -30 logs/dkt_f0_s42_latefusion.log

CKPT=sm_dkt/peiyou_dkt_qid_sm_dkt_42_0_0.2_200_0.001_0_0
               total        used        free      shared  buff/cache   available
Mem:            31Gi       1.3Gi       1.4Gi       1.3Mi        29Gi        30Gi
09:28:30 RAM used: 1G / 31G
09:28:45 RAM used: 3G / 31G
09:29:00 RAM used: 4G / 31G
09:29:15 RAM used: 5G / 31G
09:29:30 RAM used: 7G / 31G
09:29:45 RAM used: 8G / 31G
09:30:00 RAM used: 9G / 31G
09:30:15 RAM used: 11G / 31G
09:30:30 RAM used: 12G / 31G
09:30:45 RAM used: 13G / 31G
09:31:00 RAM used: 14G / 31G
09:31:15 RAM used: 16G / 31G
09:31:30 RAM used: 17G / 31G
09:31:45 RAM used: 18G / 31G
09:32:00 RAM used: 17G / 31G
09:32:15 RAM used: 18G / 31G
09:32:30 RAM used: 9G / 31G
09:32:45 RAM used: 9G / 31G
09:33:00 RAM used: 9G / 31G
09:33:15 RAM used: 10G / 31G
09:33:30 RAM used: 11G / 31G
09:33:45 RAM used: 11G / 31G
09:34:00 RAM used: 12G / 31G
09:34:15 RAM used: 12G / 31G
09:34:30 RAM used: 13G / 31G
09:34:45 RAM used: 13G / 31G
09:35:00 RAM used: 13G / 31G
09:35:15

bash: line 10:   178 Killed                  python -u wandb_predict.py --save_dir=$CKPT --bz=64 --use_wandb=0 --fusion_type=late_fusion > logs/dkt_f0_s42_latefusion.log 2>&1
